## Neo4j Query Helpers — Part 2

This notebook runs the query functions from `db/neo4j_queries.py` to verify the Neo4j drug-safety graph and explore interactions and side effects.

**Prerequisites:**
- Neo4j running (e.g. `neo4j://127.0.0.1:7687`)
- Graph loaded: Drug nodes (from RxNav/DrugBank ETL) and SIDER side effects (`etl/load_sider_to_neo4j.py`)

**Functions used:**
- `get_drug_stats()` — graph counts
- `get_side_effects(drug_name)` — side effects for a drug
- `check_interactions(current_meds, proposed_drug)` — interactions between current meds and proposed drug
- `find_shared_side_effects(drug_a, drug_b)` — side effects common to two drugs
- `find_safer_alternatives(drug_name, current_meds)` — alternatives with fewer conflicts
- `find_interaction_path(drug_a, drug_b)` — shortest interaction path
- `get_interaction_network(drug_name, depth)` — neighborhood for visualization

In [1]:
import sys
from pathlib import Path

# Ensure project root is on path (notebook lives in db/)
root = Path.cwd() if (Path.cwd() / "db").is_dir() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
import os
from getpass import getpass

# Connection settings (override with env NEO4J_URI / NEO4J_PASSWORD if you prefer)
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
if NEO4J_PASSWORD is None:
    NEO4J_PASSWORD = getpass("Neo4j password: ")

conn = {
    "uri": NEO4J_URI,
    "user": NEO4J_USER,
    "password": NEO4J_PASSWORD,
}
print(f"Using URI: {NEO4J_URI}")

Using URI: neo4j://127.0.0.1:7687


In [3]:
from db.neo4j_queries import (
    get_drug_stats,
    get_side_effects,
    check_interactions,
    find_shared_side_effects,
    find_safer_alternatives,
    find_interaction_path,
    get_interaction_network,
)

### 1. Graph statistics

In [4]:
stats = get_drug_stats(**conn)
for k, v in (stats or {}).items():
    print(f"  {k}: {v}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Drug` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=26, offset=26>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 26, 'line': 2, 'column': 26}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                MATCH (d:Drug)\n                OPTIONAL MATCH (d)-[i:INTERACTS_WITH]-()\n                OPTIONAL MATCH (d)-[h:HAS_SIDE_EFFECT]->()\n                WITH count(DISTINCT d) AS total_drugs,\n                     count(DISTINCT i) AS total_interactions,\n                     count(DISTINCT h) AS total_side_effe

### 2. Side effects for a drug

In [5]:
drug = "Warfarin"  # change to a drug name that exists in your graph
effects = get_side_effects(drug, **conn)
if effects:
    import pandas as pd
    display(pd.DataFrame(effects).head(20))
else:
    print(f"No side effects found for '{drug}' (or drug not in graph).")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_SIDE_EFFECT` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=35, offset=35>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 35, 'line': 2, 'column': 35}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n                MATCH (d:Drug)-[r:HAS_SIDE_EFFECT]->(se:SideEffect)\n                WHERE toLower(trim(d.name)) = toLower(trim($drug_name))\n                RETURN DISTINCT\n                    se.name AS side_effect,\n                    coalesce(r.frequency, 'unknown') AS frequency\n    

No side effects found for 'Warfarin' (or drug not in graph).


### 3. Interactions: current meds vs proposed drug

In [6]:
current_meds = ["Aspirin"]  # list of current medications
proposed_drug = "Warfarin"
interactions = check_interactions(current_meds, proposed_drug, **conn)
if interactions:
    import pandas as pd
    display(pd.DataFrame(interactions))
else:
    print(f"No interactions found between {current_meds} and {proposed_drug}.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=24, offset=251>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 251, 'line': 6, 'column': 24}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n                MATCH (d1:Drug)-[r:INTERACTS_WITH]-(d2:Drug)\n                WHERE toLower(trim(d1.name)) IN $current_meds\n                  AND toLower(trim(d2.name)) = toLower(trim($proposed))\n                RETURN DISTINCT\n                    d1.name AS current_drug,\n                    d2.name AS propos

No interactions found between ['Aspirin'] and Warfarin.


### 4. Shared side effects between two drugs

In [7]:
drug_a, drug_b = "Warfarin", "Aspirin"
shared = find_shared_side_effects(drug_a, drug_b, **conn)
if shared:
    import pandas as pd
    display(pd.DataFrame(shared).head(15))
else:
    print(f"No shared side effects for {drug_a} and {drug_b}.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_SIDE_EFFECT` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=71, offset=71>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 71, 'line': 2, 'column': 71}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                MATCH (a:Drug)-[:HAS_SIDE_EFFECT]->(se:SideEffect)<-[:HAS_SIDE_EFFECT]-(b:Drug)\n                WHERE toLower(trim(a.name)) = toLower(trim($drug_a))\n                  AND toLower(trim(b.name)) = toLower(trim($drug_b))\n                RETURN DISTINCT\n                    s

No shared side effects for Warfarin and Aspirin.


### 5. Safer alternatives (same “family”, fewer conflicts with current meds)

In [8]:
proposed = "Warfarin"
current = ["Aspirin"]
alts = find_safer_alternatives(proposed, current, **conn)
if alts:
    import pandas as pd
    display(pd.DataFrame(alts))
else:
    print(f"No alternatives found for {proposed} given current meds {current}.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `HAS_SIDE_EFFECT` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=78, offset=78>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 78, 'line': 2, 'column': 78}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                MATCH (proposed:Drug)-[:HAS_SIDE_EFFECT]->(se:SideEffect)<-[:HAS_SIDE_EFFECT]-(alt:Drug)\n                WHERE toLower(trim(proposed.name)) = toLower(trim($drug_name))\n                  AND proposed <> alt\n                WITH alt, count(DISTINCT se) AS shared_se_count\n 

No alternatives found for Warfarin given current meds ['Aspirin'].


### 6. Shortest interaction path between two drugs

In [9]:
path_a, path_b = "Warfarin", "Ibuprofen"
paths = find_interaction_path(path_a, path_b, **conn)
if paths:
    for p in paths:
        print(" -> ".join(p["path_drugs"]), f"(hops: {p['path_length']})")
else:
    print(f"No interaction path found between {path_a} and {path_b}.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=46, offset=300>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 300, 'line': 6, 'column': 46}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                MATCH (a:Drug), (b:Drug)\n                WHERE toLower(trim(a.name)) = toLower(trim($drug_a))\n                  AND toLower(trim(b.name)) = toLower(trim($drug_b))\n                MATCH path = shortestPath((a)-[:INTERACTS_WITH*1..3]->(b))\n                RETURN [n IN nodes(path) | n.name] AS pa

No interaction path found between Warfarin and Ibuprofen.


### 7. Interaction network around a drug (for visualization)

In [10]:
center_drug = "Warfarin"
depth = 2
net = get_interaction_network(center_drug, depth=depth, **conn)
print(f"Nodes: {len(net['nodes'])}, Edges: {len(net['edges'])}")
if net["nodes"]:
    print("Node names:", [n["name"] for n in net["nodes"][:15]])
if net["edges"]:
    print("Sample edges:", net["edges"][:5])

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `name` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=43, offset=79>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 79, 'line': 3, 'column': 43}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n                MATCH (center:Drug)\n                WHERE toLower(trim(center.name)) = toLower(trim($drug_name))\n                MATCH path = (center)-[:INTERACTS_WITH*1..2]-(neighbor:Drug)\n                WITH center, collect(DISTINCT neighbor) AS neighbors\n                WITH [center] + neighbors AS all_node

Nodes: 0, Edges: 0
